In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.agents import AgentState

class CustomState(AgentState):
    favourite_colour: str

## Write to state

In [3]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(update={
        "favourite_colour": favourite_colour, 
        "messages": [ToolMessage("Successfully updated favourite colour", tool_call_id=runtime.tool_call_id)]}
        )

In [4]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    "google_genai:gemini-3.5-flash-lite",
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [5]:
from langchain.messages import HumanMessage

response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

In [9]:
from pprint import pprint

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}, id='350daf13-0087-4bb4-9242-561df278fc74'),
              AIMessage(content=[{'type': 'text', 'text': "Hello! I'm doing well, thank you for asking. How can I help you today?", 'extras': {'signature': 'EmAKXgFpFH0TthjvFzqJdiUhq6aH4iKh9Sq1pus1EjQTAduGh2aqNokzUlxTc6G6YX+ZttyJ/cwLAR8Pn5L2s8uyM35wQb5DUdymcbE/4qwBPpVowv0gEtu+aVnA2pJHJJ4='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1cdfd-ce4b-7593-ae99-7e0e56bf1a82-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 70, 'output_tokens': 20, 'total_tokens': 90, 'input_token_details': {'cache_read': 0}}),
              HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}, id='d0abdaaf-1b9e-4335-9be4-d921c84a7f6b'),
         

In [8]:
response = agent.invoke(
    { 
        "messages": [HumanMessage(content="Hello, how are you?")],
        "favourite_colour": "green"
    },
    {"configurable": {"thread_id": "10"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}, id='350daf13-0087-4bb4-9242-561df278fc74'),
              AIMessage(content=[{'type': 'text', 'text': "Hello! I'm doing well, thank you for asking. How can I help you today?", 'extras': {'signature': 'EmAKXgFpFH0TthjvFzqJdiUhq6aH4iKh9Sq1pus1EjQTAduGh2aqNokzUlxTc6G6YX+ZttyJ/cwLAR8Pn5L2s8uyM35wQb5DUdymcbE/4qwBPpVowv0gEtu+aVnA2pJHJJ4='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1cdfd-ce4b-7593-ae99-7e0e56bf1a82-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 70, 'output_tokens': 20, 'total_tokens': 90, 'input_token_details': {'cache_read': 0}}),
              HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}, id='d0abdaaf-1b9e-4335-9be4-d921c84a7f6b'),
         

## Read state

In [10]:
@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"

agent = create_agent(
    "google_genai:gemini-3.5-flash-lite",
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [11]:
response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='ff10ba9b-0ebb-41e8-9aa3-091f63a59376'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'update_favourite_colour', 'arguments': '{"favourite_colour": "green"}'}, '__gemini_function_call_thought_signatures__': {'call_15806359588539007715': 'EmAKXgFpFH0TmKMAU3YbGzlKwmFIAp1GfkaN2NeteH3KoaPxxhuMr/QJ1jXtCmRQ0+IUZYLRCJFn7TYKfBGN/UA8GI10EuKO0DhgGw2HpR41K+v6aivGq6XjydSvTsoMHy4='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1cdff-5efb-7132-af27-23e59aaa4e0a-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'call_15806359588539007715', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 96, 'output_tokens': 22, 'total_tokens': 118, 'input

In [13]:
response = agent.invoke(
    { "messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response["messages"][-1].content[0]['text'])

'Your favourite colour is green!'
